# EventTCN comparison — news-event conditioning, two hyperparameter sets, and HAR-RV

Five model families over all three horizons (h = 1 daily, 5 weekly, 22 monthly), on
`data/realized_volatility.csv` plus the EUR/USD economic calendar in
`data/eurusd_calendar_events_2010_2025 (1).csv`.

| # | model | data | hyperparameters |
|---|---|---|---|
| 1 | **HAR-RV** | RV only | OLS, nothing to tune |
| 2 | **ModernTCN (A)** | RV only | *A* — this repo's Optuna best (`scripts/RV.sh`) |
| 3 | **EventTCN (A)** | RV + calendar | *A* |
| 4 | **ModernTCN (B)** | RV only | *B* — the FilmTCN repo's Optuna best |
| 5 | **EventTCN (B)** | RV + calendar | *B* |

Rows 2–3 and 4–5 are matched pairs: within a backbone the **only** difference is whether the
model sees the calendar, so each pair isolates the event effect, and the two pairs together
say how much that effect depends on the backbone.

### Everything below forecasts the same object

$$Y_t^{(h)} = \ln\!\left(\tfrac{1}{h}\sum_{k=1}^{h} RV_{t+k}\right)$$

one number per origin — the log of the **arithmetic** mean RV over the next *h* days, log
outside the sum. HAR-RV is run with `--log`, which uses the same convention, so its losses sit
in the same column as the neural ones.

> This is **not** the target the FilmTCN repo optimises. There the aggregation is a mean of
> $\ln RV$ — the log of a *geometric* mean, a smaller and much smoother object. The two
> coincide only at h = 1. So the numbers in this notebook are not comparable to that repo's
> `model_comparison_table.md`, and neither is its HAR-RV column.

### Splits and scored rows

From `data_provider/splits.py`, which every family reads:

| | train | validation | test |
|---|---|---|---|
| neural | 2010-01-01 – 2021-12-31 | 2022-01-01 – 2023-12-31 | 2024-01-01 – 2025-04-07 |
| HAR-RV | 2010-01-01 – 2023-12-31 (train + val) | — | 2024-01-01 – 2025-04-07 |

Every test window is scored — **328 / 324 / 307** origins at h = 1 / 5 / 22, the h−1 shrinkage
being the embargo. The count does not depend on `seq_len`, so all five families are scored on
identical rows.

### What the event conditioning is

The calendar carries no actual, forecast or previous value — only that an event is *scheduled*,
with its currency and impact rating. So the horizon slice is a genuine known-in-advance
covariate: the model learns that an FOMC statement lands inside the window being forecast,
never what it said. Past events enter through their own patch stem; future events are pooled
into a zero-initialised FiLM `(gamma, beta)` over the final feature map. Neither path touches
RevIN. See the `EventTCN` section of the README.


## 1. Setup


In [ ]:
import os, sys, subprocess

REPO   = "https://github.com/Mr0022/ModernTCNt.git"
BRANCH = "claude/moderntcn-aggregation-log-ptj8ao"
ROOT   = "/content/ModernTCNt"

if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO, ROOT], check=True)
else:
    print("Repository already cloned; pulling the latest commit.")
    subprocess.run(["git", "-C", ROOT, "pull", "--ff-only"], check=False)

# os.chdir, not %cd: it moves the Python process, so the ! cells below inherit it.
os.chdir(os.path.join(ROOT, "ModernTCN-Long-term-forecasting"))
print("working directory:", os.getcwd())

# Colab ships torch, pandas, numpy, sklearn, matplotlib and scipy. statsmodels is
# what HAR-RV needs for its OLS and HAC covariance; install only if it is missing.
try:
    import statsmodels  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "statsmodels"], check=True)

import torch, pandas as pd
print("torch", torch.__version__, "| GPU:",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)")
print("RV rows:", len(pd.read_csv("data/realized_volatility.csv")))
print("calendar rows:", len(pd.read_csv("data/eurusd_calendar_events_2010_2025 (1).csv")))


## 2. Configuration

Two hyperparameter sets, both taken verbatim from Optuna studies. **Where they came from
matters, because the provenance is not what this repo's script header claims.**

**A — this repo's `scripts/RV.sh`.** These are the values `RV.sh` and the README present as the
per-horizon search result. They are in fact the FilmTCN repo's **EventTCN** studies
(`tuningresults/EVENTTCN{1,5,22}/best_params.json`) — i.e. they were selected with an
event-conditioned model in the loop, then reused here for the unconditioned baseline. The
objective values match to seven decimals (0.17672090, 0.08440566, 0.06684236), as do all the
parameters. Worth fixing in the write-up; it is reproduced faithfully here so the table shows
what the repo actually ships.

**B — the FilmTCN repo's plain-model studies** (`tuningresults/ModernTCN{1,5,22}`), searched on
ModernTCN *without* events. Larger models: `dims` 32 / 128 / 256 and much smaller learning
rates at h = 5 and 22.

`event_dim` is a searched hyperparameter, so each backbone carries its own. Everything else
about the conditioning is held fixed across all event runs — role vocabulary, channel fusion,
both paths on — so the table varies the backbone, not the method.


In [ ]:
HORIZONS = [1, 5, 22]     # h = daily, weekly, monthly
SEEDS    = 5              # --itr: seeds 2021 .. 2025
DES      = "Cmp"          # tags the output filenames

# Held fixed across every event run, so the comparison varies the backbone only.
EVENT_VOCAB  = "role"     # collapses 'FOMC Member <person> Speaks' etc. to one column
EVENT_FUSION = "channel"  # past events as a backbone variable (FilmTCN's tuned default)

BACKBONE = {
    "A": dict(
        label="ModernTCNt Optuna best (scripts/RV.sh)",
        train=" --lradj type3 --train_epochs 50 --patience 10",
        cfg={
            1:  dict(seq_len=22, patch_size=32, patch_stride=8, ffn_ratio=3, large_size=51,
                     small_size=3, num_blocks=3, dropout=0.4155, head_dropout=0.2993,
                     learning_rate=0.00550, batch_size=128, dim=32, event_dim=16),
            5:  dict(seq_len=35, patch_size=32, patch_stride=2, ffn_ratio=3, large_size=13,
                     small_size=5, num_blocks=3, dropout=0.0751, head_dropout=0.3690,
                     learning_rate=0.00431, batch_size=128, dim=32, event_dim=4),
            22: dict(seq_len=35, patch_size=4,  patch_stride=8, ffn_ratio=2, large_size=31,
                     small_size=5, num_blocks=1, dropout=0.2332, head_dropout=0.3228,
                     learning_rate=0.00779, batch_size=256, dim=32, event_dim=8),
        }),
    "B": dict(
        label="FilmTCN Optuna best (plain ModernTCN studies)",
        train=" --lradj TST --pct_start 0.3 --train_epochs 40 --patience 8",
        cfg={
            1:  dict(seq_len=70, patch_size=16, patch_stride=8, ffn_ratio=2, large_size=27,
                     small_size=5, num_blocks=2, dropout=0.33157505058759384,
                     head_dropout=0.13413677333143775, learning_rate=0.0063484758647924695,
                     batch_size=256, dim=32, event_dim=16),
            5:  dict(seq_len=22, patch_size=16, patch_stride=8, ffn_ratio=1, large_size=13,
                     small_size=3, num_blocks=1, dropout=0.4744918542935045,
                     head_dropout=0.16435589854180058, learning_rate=9.048320833685613e-05,
                     batch_size=256, dim=128, event_dim=16),
            22: dict(seq_len=22, patch_size=16, patch_stride=2, ffn_ratio=3, large_size=51,
                     small_size=7, num_blocks=1, dropout=0.3222005482423507,
                     head_dropout=0.18550313066719137, learning_rate=0.0001385051157761346,
                     batch_size=256, dim=256, event_dim=16),
        }),
}

# name -> (backbone key, uses events); HAR-RV is handled separately.
MODELS = {
    "ModernTCN (A)": ("A", False),
    "EventTCN (A)":  ("A", True),
    "ModernTCN (B)": ("B", False),
    "EventTCN (B)":  ("B", True),
}

def tag_of(bk, events, h):
    return f"{bk}_{'ev' if events else 'noev'}_h{h}"

def build_cmd(bk, events, h):
    b = BACKBONE[bk]; c = b["cfg"][h]; d = c["dim"]
    cmd = (
        "python -u run.py --is_training 1 --model ModernTCN"
        f" --model_id {tag_of(bk, events, h)} --des {DES}"
        f" --data {'RVEvents' if events else 'RV'}"
        " --root_path ./data/ --data_path realized_volatility.csv"
        " --features S --target RV --asset forex --freq d --enc_in 1 --revin 1"
        f" --seq_len {c['seq_len']} --label_len 0 --pred_len {h}"
        f" --patch_size {c['patch_size']} --patch_stride {c['patch_stride']}"
        f" --ffn_ratio {c['ffn_ratio']} --large_size {c['large_size']}"
        f" --small_size {c['small_size']} --num_blocks {c['num_blocks']}"
        f" --dims {d} {d} {d} {d} --dw_dims {d} {d} {d} {d}"
        f" --dropout {c['dropout']} --head_dropout {c['head_dropout']}"
        f" --learning_rate {c['learning_rate']} --batch_size {c['batch_size']}"
        f" --random_seed 2021 --itr {SEEDS}" + b["train"] +
        " --use_multi_scale False --small_kernel_merged False --num_workers 2"
    )
    if events:
        cmd += (f" --event_vocab {EVENT_VOCAB} --event_fusion {EVENT_FUSION}"
                f" --event_dim {c['event_dim']}")
    return cmd

print(f"{len(MODELS) * len(HORIZONS)} configurations x {SEEDS} seeds "
      f"= {len(MODELS) * len(HORIZONS) * SEEDS} training runs, plus one HAR-RV fit.")


## 3. HAR-RV — Corsi (2009), log scale

One invocation fits all three horizons. `--log` puts the whole pipeline on the ln(RV) scale with
the log-of-mean convention; `--asset forex` selects the split calendar. OLS has nothing to tune,
so there is nothing to seed and nothing to average.


In [ ]:
!python -u HAR-RV_RUN.PY --data data/realized_volatility.csv --log --asset forex --outdir results_har


## 4. The twelve neural runs

Four model families x three horizons, each over `SEEDS` seeds trained from scratch. Per-seed
MSE / MAE / QLIKE print as each run goes, followed by the mean and standard deviation.

On a Colab T4 the whole cell is roughly 30–60 minutes; the B backbone at h = 5 and 22 is the
slow part (`dims` 128 and 256). It is safe to re-run — each configuration overwrites its own
files and nothing is shared between them.


In [ ]:
for name, (bk, events) in MODELS.items():
    for h in HORIZONS:
        c = BACKBONE[bk]["cfg"][h]
        print("\n" + "#" * 78)
        print(f"#  {name}  h={h}  --  {SEEDS} seeds, seq_len={c['seq_len']}, dim={c['dim']}"
              + (f", event_dim={c['event_dim']}" if events else ""))
        print(f"#  {BACKBONE[bk]['label']}")
        print("#" * 78)
        cmd = build_cmd(bk, events, h)
        # !{cmd} runs the string in a subshell and streams its output into the cell.
        !{cmd}


## 5. Results

`MSE` and `MAE` are in **ln(RV)** units — the scale every family is fitted on. `QLIKE`
(Patton, 2011) needs variances, so it is evaluated after `exp()` with the lognormal Jensen
correction $E[RV\mid\mathcal F] = \exp(E[\ln RV\mid\mathcal F] + \sigma^2/2)$; `MSE_RV` and
`MAE_RV` are the same forecasts on that back-transformed scale. Neural columns are the mean
over seeds ± the standard deviation. Lower is better throughout.


In [ ]:
import pandas as pd, numpy as np
from IPython.display import display

METRICS = ["MSE", "MAE", "QLIKE", "MSE_RV", "MAE_RV"]

har = pd.read_csv("results_har/har_rv_log_all_metrics.csv")
har = har[har["split"] == "test"].set_index("horizon")

rows, missing = [], []
for h in HORIZONS:
    for m in METRICS:
        rows.append(dict(horizon=h, metric=m, model="HAR-RV",
                         value=float(har.loc[h, m]), sd=np.nan, seeds=0))
    for name, (bk, events) in MODELS.items():
        path = f"results/{tag_of(bk, events, h)}_{DES}_seed_metrics.csv"
        try:
            d = pd.read_csv(path)
        except FileNotFoundError:
            missing.append(path)
            continue
        d["seed"] = d["seed"].astype(str)
        mean = d[d.seed == "mean"].iloc[0]
        sd   = d[d.seed == "std"].iloc[0]
        n    = int((~d.seed.isin(["mean", "std"])).sum())
        for m in METRICS:
            rows.append(dict(horizon=h, metric=m, model=name,
                             value=float(mean[m]), sd=float(sd[m]), seeds=n))

res = pd.DataFrame(rows)
if missing:
    print("MISSING (re-run section 4 for these):")
    for p in missing:
        print("  ", p)

ORDER = ["HAR-RV"] + list(MODELS)

def cell(r):
    return f"{r.value:.4f}" if np.isnan(r.sd) else f"{r.value:.4f} ± {r.sd:.4f}"

pretty = (res.assign(txt=res.apply(cell, axis=1))
             .pivot(index=["metric", "model"], columns="horizon", values="txt")
             .reindex(pd.MultiIndex.from_product([METRICS, ORDER], names=["metric", "model"]))
             .dropna(how="all"))
pretty.columns = [f"h={h}" for h in pretty.columns]

n_seeds = int(res[res.model != "HAR-RV"].seeds.max())
print("Test window 2024-01-01 .. 2025-04-07 -- 328 / 324 / 307 origins at h = 1 / 5 / 22,")
print(f"identical rows for every family. Neural rows are the mean over {n_seeds} seeds.\n")
display(pretty)

res.to_csv("results/eventtcn_comparison.csv", index=False)
print("\nSaved: results/eventtcn_comparison.csv")


### The event effect, paired

Within a backbone the two arms differ only in whether the model sees the calendar, so this is
the quantity the whole notebook exists to measure. Negative Δ% means the conditioned model is
better.

Watch the horizon gradient. Over this calendar the forward high-impact event count has a
coefficient of variation of **0.92 at h = 1, 0.50 at h = 5 and 0.27 at h = 22**, and no h = 22
window is event-free at all — by the monthly horizon nearly every window looks alike, so there
is very little left to condition on. A *large* h = 22 gain would be a reason to look for a leak,
not to celebrate. Compare each Δ% against the seed standard deviations in the table above: a
gap smaller than the spread is not a result.


In [ ]:
pairs = [("A", "ModernTCN (A)", "EventTCN (A)"), ("B", "ModernTCN (B)", "EventTCN (B)")]
gap = []
for bk, base, ev in pairs:
    for h in HORIZONS:
        for m in METRICS:
            sel = res[(res.horizon == h) & (res.metric == m)]
            b = sel[sel.model == base]
            e = sel[sel.model == ev]
            if b.empty or e.empty:
                continue
            b, e = float(b.value.iloc[0]), float(e.value.iloc[0])
            gap.append(dict(backbone=f"{bk} -- {BACKBONE[bk]['label']}", horizon=h,
                            metric=m, no_events=b, events=e, delta_pct=100 * (e - b) / b))

gap = pd.DataFrame(gap)
order = pd.MultiIndex.from_product([gap.backbone.unique(), METRICS],
                                   names=["backbone", "metric"])
show = (gap.assign(txt=gap.delta_pct.map("{:+.1f}%".format))
           .pivot(index=["backbone", "metric"], columns="horizon", values="txt")
           .reindex(order).dropna(how="all"))
show.columns = [f"h={h}" for h in show.columns]
print("Event effect: (EventTCN - ModernTCN) / ModernTCN, same backbone. Negative = events help.\n")
display(show)
gap.to_csv("results/eventtcn_event_effect.csv", index=False)
print("\nSaved: results/eventtcn_event_effect.csv")


### Per-seed detail

One seed is a draw, not a result. The spread here is what says whether a gap in the tables above
is real.


In [ ]:
for h in HORIZONS:
    print("\n" + "=" * 78)
    print(f"  h = {h}")
    print("=" * 78)
    for name, (bk, events) in MODELS.items():
        path = f"results/{tag_of(bk, events, h)}_{DES}_seed_metrics.csv"
        try:
            d = pd.read_csv(path)
        except FileNotFoundError:
            print(f"\n--- {name}: not run")
            continue
        print(f"\n--- {name}")
        display(d[["seed", "MSE", "MAE", "QLIKE", "MSE_RV", "MAE_RV"]]
                .set_index("seed").round(6))


### LaTeX table

Booktabs, ready to paste — one row per model, three metrics per horizon. Numbers are plain; swap
in your `\raa{}{}` macro if the document needs the RTL decimal form.


In [ ]:
HEAD = ["MSE", "MAE", "QLIKE"]

lines = [r"\begin{table}[htbp]", r"\centering", r"\small",
         r"\caption{Out-of-sample forecast losses on the aggregated log target "
         r"$Y_t^{(h)}=\ln(\frac{1}{h}\sum_{k=1}^{h}RV_{t+k})$, test window "
         r"2024-01-01--2025-04-07 (328/324/307 origins at $h=1/5/22$). Neural rows are the "
         r"mean over %d seeds. MSE and MAE are in $\ln(RV)$ units; QLIKE is on the "
         r"back-transformed variance scale. Lower is better.}" % n_seeds,
         r"\label{tab:eventtcn-comparison}",
         r"\begin{tabular}{l" + "ccc" * len(HORIZONS) + "}", r"\toprule",
         r"& " + " & ".join(r"\multicolumn{3}{c}{$h=%d$}" % h for h in HORIZONS) + r" \\",
         r"\textbf{Model} & " + " & ".join(HEAD * len(HORIZONS)) + r" \\",
         r"\midrule"]
for model in ORDER:
    cells_ = []
    for h in HORIZONS:
        for m in HEAD:
            sel = res[(res.horizon == h) & (res.metric == m) & (res.model == model)]
            cells_.append(f"{float(sel.value.iloc[0]):.4f}" if not sel.empty else "--")
    lines.append(model.replace("_", r"\_") + " & " + " & ".join(cells_) + r" \\")
    if model in ("HAR-RV", "EventTCN (A)"):
        lines.append(r"\midrule")
lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]

latex = "\n".join(lines)
print(latex)
with open("results/eventtcn_comparison_table.tex", "w") as f:
    f.write(latex)
print("\nSaved: results/eventtcn_comparison_table.tex")


## 6. Where the outputs are

| path | what |
|---|---|
| `results/{A,B}_{noev,ev}_h{h}_Cmp_seed_metrics.csv` | one row per seed plus mean and std |
| `results/<setting>/rv_forecasts.csv` | per-origin forecast, dated by the first day of the target window |
| `results/<setting>/rv_metrics.csv` | one run's losses, HAR-RV's column names |
| `results/eventtcn_comparison.csv` | the main table |
| `results/eventtcn_event_effect.csv` | the paired event effect |
| `results/eventtcn_comparison_table.tex` | the LaTeX table |
| `results_har/har_rv_log_all_metrics.csv` | HAR-RV losses, all horizons |
| `results_har/har_rv_log_h{h}_params.csv` | HAR-RV coefficients, std errors, HAC t-stats |

Colab discards `/content` when the runtime ends. To keep the results, download them or mount
Drive and copy `results/` and `results_har/` across.

### Two caveats for the write-up

**The comparator for an event gain is HAR-X, not HAR-RV.** HAR-RV with announcement counts as
extra regressors is the model that shows whether *FiLM conditioning* helps, rather than merely
that the calendar is informative. It is not in this table because it is not yet implemented.

**The horizon pool is a mean.** `event_embed(event_y).mean(dim=1)` is order-free, so a release
on day 1 of the window and one on day h condition the model identically. A sum with learned
per-position weights is the obvious next experiment and the likeliest reason a long-horizon
effect would fail to appear.
